In [ ]:
# Client Setupfrom dotenv import load_dotenvimport voyageaiload_dotenv()client = voyageai.Client()

In [ ]:
# Chunkingimport redef chunk_by_section(document_text):    return re.split(r"\n## ", document_text)

In [ ]:
# Embeddingdef generate_embedding(chunks, model="voyage-3-large", input_type="query"):    is_list = isinstance(chunks, list)    data = chunks if is_list else [chunks]    result = client.embed(data, model=model, input_type=input_type)    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
# Simplified VectorIndex (core only)class VectorIndex:    def __init__(self, embedding_fn):        self.embedding_fn = embedding_fn        self.vectors = []        self.docs = []    def add_documents(self, docs):        texts = [d['content'] for d in docs]        embs = self.embedding_fn(texts)        self.vectors.extend(embs)        self.docs.extend(docs)    def search(self, query, k=3):        q = self.embedding_fn(query)        scores = []        for i, v in enumerate(self.vectors):            score = sum(a*b for a,b in zip(q, v))            scores.append((self.docs[i], score))        return sorted(scores, key=lambda x: x[1], reverse=True)[:k]

In [ ]:
# BM25 (simplified)from collections import Counterimport mathclass BM25Index:    def __init__(self):        self.docs = []        self.tokens = []    def add_documents(self, docs):        for d in docs:            t = d['content'].lower().split()            self.docs.append(d)            self.tokens.append(t)    def search(self, query, k=3):        q_tokens = query.lower().split()        scores = []        for i, doc_tokens in enumerate(self.tokens):            score = sum(doc_tokens.count(t) for t in q_tokens)            scores.append((self.docs[i], score))        return sorted(scores, key=lambda x: x[1], reverse=True)[:k]

In [ ]:
# Retriever with RRFclass Retriever:    def __init__(self, *indexes):        self.indexes = indexes    def add_documents(self, docs):        for idx in self.indexes:            idx.add_documents(docs)    def search(self, query, k=3, k_rrf=60):        results = [idx.search(query, k=10) for idx in self.indexes]        rank_map = {}        for i, res in enumerate(results):            for r, (doc, _) in enumerate(res):                key = id(doc)                if key not in rank_map:                    rank_map[key] = {"doc": doc, "ranks": [float('inf')]*len(self.indexes)}                rank_map[key]["ranks"][i] = r+1        def rrf(ranks):            return sum(1/(k_rrf+r) for r in ranks if r != float('inf'))        scored = [(v["doc"], rrf(v["ranks"])) for v in rank_map.values()]        return sorted(scored, key=lambda x: x[1], reverse=True)[:k]

In [ ]:
# Demo pipelinewith open('report.md') as f:    text = f.read()chunks = chunk_by_section(text)docs = [{"content": c} for c in chunks]vec = VectorIndex(generate_embedding)bm25 = BM25Index()retriever = Retriever(vec, bm25)retriever.add_documents(docs)print(retriever.search("INC-2023-Q4-011", k=3))